# Receiver-Causal Replication — Kaggle T4×2

**Version:** `RECEIVER_CAUSAL_T4X2_V3`

Run this in a **fresh Kaggle session** with **GPU T4 ×2** and **Internet ON**.

This notebook is designed around the remaining ~16–18 GPU-hours and Kaggle's tight working-disk budget:

1. **Required:** Mistral-7B-v0.3 full replication at the primary depth:
   - 256 C4 contexts,
   - 65,280 directed host×donor pairs,
   - 512 fixed natural displacements,
   - 16 physical receivers per displacement.

2. **Required:** physical fixed-$\Delta u$ receiver crossing across four depths in both Mistral-7B and Qwen3-8B, without recomputing a 65k pair matrix at every depth.

3. **Recommended:** repeat the primary-depth physical receiver-crossing test on two additional disjoint C4 buckets using 32 receivers per fixed displacement.

4. **Deliberately deferred:** exact JVP. It already OOMed twice on T4×2; do that later on a larger-memory GPU.

### Storage rule
Only **one model family is cached at a time**. The HF model cache is deleted between models. Dataset cache is deleted after the tiny deterministic C4 pools are saved.


**V3 loader fix:** uses `device_map="balanced"` with `12000MiB` max per T4, adds a Mistral load-only smoke test, and reuses the successfully loaded model for Phase 1.

In [ ]:
# ============================================================
# ENVIRONMENT CHECK — DO NOT UPGRADE PACKAGES UNLESS MISSING
# ============================================================
import os, sys, subprocess, importlib, gc, json, math, time, random, hashlib, shutil, zipfile, traceback, warnings, re
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict

required_imports = {
    "transformers": "transformers>=4.51,<5",
    "accelerate": "accelerate>=1.6,<2",
    "datasets": "datasets>=3.0,<4",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
}
missing = []
for mod, spec in required_imports.items():
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(spec)

if missing:
    print("Installing only missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Required packages already available; no pip changes made.")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers, accelerate, datasets
from tqdm.auto import tqdm
from scipy.stats import spearmanr, wilcoxon, rankdata
from sklearn.decomposition import PCA
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("datasets:", datasets.__version__)


In [ ]:
# ============================================================
# CONFIG + FRESH T4x2 PREFLIGHT
# ============================================================
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

warnings.filterwarnings("ignore")
torch.set_grad_enabled(False)

VERSION = "RECEIVER_CAUSAL_T4X2_V3"
OUT_ROOT = Path(f"/kaggle/working/{VERSION}")
MODEL_CACHE = Path("/kaggle/working/hf_model_cache")
DATA_CACHE = Path("/kaggle/working/hf_dataset_cache")
OFFLOAD_DIR = Path("/kaggle/working/hf_offload")
CHECKPOINT_ZIP = Path(f"/kaggle/working/{VERSION}_CHECKPOINT.zip")
FINAL_ZIP = Path(f"/kaggle/working/{VERSION}_RESULTS.zip")

for p in [OUT_ROOT, MODEL_CACHE, DATA_CACHE, OFFLOAD_DIR]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(DATA_CACHE)

@dataclass
class CFG:
    seed: int = 260826
    n_contexts: int = 256
    raw_pool_size: int = 320
    core_tokens: int = 64
    c4_modulus: int = 11
    buckets: tuple = (1, 2, 3)

    pair_batch: int = 4
    stable_eps: float = 0.075
    rel_depths: tuple = (0.20, 0.40, 0.60, 0.74)

    n_cross_deltas: int = 512
    receivers_bucket1: int = 16
    receivers_replication: int = 32

    permutation_reps: int = 2000
    resume: bool = True
    cleanup_completed_chunks: bool = True

    # Leaves a few hours of weekly-GPU buffer.
    soft_wallclock_hours: float = 15.0

cfg = CFG()

MODEL_SPECS = {
    "mistral": {
        "model_id": "mistralai/Mistral-7B-v0.3",
        "expected_layers": 32,
        "primary_expected_layer": 12,
    },
    "qwen": {
        "model_id": "Qwen/Qwen3-8B",
        "expected_layers": 36,
        "primary_expected_layer": 14,
    },
}

RUN_START = time.time()

assert torch.cuda.is_available(), "Kaggle GPU accelerator is OFF."
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
gpu_mem = [torch.cuda.get_device_properties(i).total_memory / 2**30 for i in range(torch.cuda.device_count())]
print("GPUs:", list(zip(gpu_names, [round(x,2) for x in gpu_mem])))

assert torch.cuda.device_count() == 2 and all("T4" in x.upper() for x in gpu_names), (
    f"Select Kaggle GPU T4 x2. Detected: {gpu_names}"
)

for i in range(2):
    alloc = torch.cuda.memory_allocated(i) / 2**30
    if alloc > 0.75:
        raise RuntimeError(
            f"GPU {i} already has {alloc:.2f} GiB allocated. "
            "Use a FRESH Kaggle session before running this notebook."
        )

MODEL_DTYPE = torch.float16
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

print("\nPRECHECK PASS — fresh T4 x2")
print("Contexts per bucket:", cfg.n_contexts)
print("Primary full matrix:", cfg.n_contexts * (cfg.n_contexts - 1))
print("Bucket-1 direct interventions/stage:", cfg.n_cross_deltas * cfg.receivers_bucket1)
print("Bucket-2/3 direct interventions/stage:", cfg.n_cross_deltas * cfg.receivers_replication)

with open(OUT_ROOT / "config.json", "w") as f:
    json.dump({"cfg": asdict(cfg), "models": MODEL_SPECS, "gpus": gpu_names}, f, indent=2, default=list)


In [ ]:
# ============================================================
# DISK / GPU / CHECKPOINT HELPERS
# ============================================================
def disk_status(label="disk"):
    total, used, free = shutil.disk_usage("/kaggle/working")
    print(f"{label}: used={used/2**30:.2f} GiB | free={free/2**30:.2f} GiB | total={total/2**30:.2f} GiB")
    return free / 2**30

def gpu_status(label="GPU"):
    print(label)
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(
            i,
            torch.cuda.get_device_name(i),
            f"allocated={torch.cuda.memory_allocated(i)/2**30:.2f} GiB",
            f"reserved={torch.cuda.memory_reserved(i)/2**30:.2f} GiB",
            f"free={free/2**30:.2f}/{total/2**30:.2f} GiB",
        )

def clear_model_cache():
    for p in [MODEL_CACHE, OFFLOAD_DIR]:
        if p.exists():
            shutil.rmtree(p, ignore_errors=True)
        p.mkdir(parents=True, exist_ok=True)

    for p in [Path("/root/.cache/huggingface/xet"), Path("/root/.cache/huggingface/hub")]:
        if p.exists():
            shutil.rmtree(p, ignore_errors=True)

    gc.collect()
    torch.cuda.empty_cache()

def clear_dataset_cache():
    if DATA_CACHE.exists():
        shutil.rmtree(DATA_CACHE, ignore_errors=True)
    DATA_CACHE.mkdir(parents=True, exist_ok=True)
    gc.collect()

def write_checkpoint_zip():
    tmp = Path(str(CHECKPOINT_ZIP) + ".tmp")
    if tmp.exists():
        tmp.unlink()
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in OUT_ROOT.rglob("*"):
            if p.is_file():
                zf.write(p, arcname=p.relative_to(OUT_ROOT))
    os.replace(tmp, CHECKPOINT_ZIP)
    print(f"Checkpoint: {CHECKPOINT_ZIP} ({CHECKPOINT_ZIP.stat().st_size/2**20:.1f} MB)")

def elapsed_hours():
    return (time.time() - RUN_START) / 3600.0

def phase_ok(estimated_hours, name):
    elapsed = elapsed_hours()
    projected = elapsed + estimated_hours
    print(f"Phase '{name}': elapsed={elapsed:.2f}h, projected={projected:.2f}h, soft wall={cfg.soft_wallclock_hours:.2f}h")
    if projected > cfg.soft_wallclock_hours:
        print("SKIP optional phase to preserve weekly-GPU buffer.")
        return False
    return True

disk_status("Initial disk")
gpu_status("Initial GPU")


In [ ]:
# ============================================================
# SELECT THREE DISJOINT C4 BUCKETS IN ONE STREAMING PASS
# ============================================================
def bucket_path(bucket):
    return OUT_ROOT / f"c4_bucket{bucket}_pool.jsonl"

def load_saved_bucket(bucket, n_docs):
    p = bucket_path(bucket)
    if not p.exists():
        return None
    rows = [json.loads(x) for x in p.read_text(encoding="utf-8").splitlines() if x.strip()]
    return rows[:n_docs] if len(rows) >= n_docs else None

def select_c4_buckets(buckets, n_docs, min_chars=2400):
    pools = {}
    missing = []
    for b in buckets:
        saved = load_saved_bucket(b, n_docs)
        if saved is not None:
            pools[b] = saved
            print(f"RESUME bucket {b}: {len(saved)} docs")
        else:
            pools[b] = []
            missing.append(b)

    if not missing:
        return pools

    ds = load_dataset(
        "allenai/c4",
        "en",
        split="validation",
        streaming=True,
        cache_dir=str(DATA_CACHE),
    )

    targets = set(missing)
    for ex in tqdm(ds, desc=f"Selecting C4 buckets {sorted(targets)}"):
        text = ex.get("text", "")
        if len(text) < min_chars:
            continue
        h = hashlib.sha256(text.encode("utf-8")).hexdigest()
        b = int(h[:8], 16) % cfg.c4_modulus
        if b not in targets or len(pools[b]) >= n_docs:
            continue
        pools[b].append({"sha256": h, "text": text})
        if all(len(pools[x]) >= n_docs for x in targets):
            break

    for b in missing:
        if len(pools[b]) < n_docs:
            raise RuntimeError(f"Bucket {b}: only {len(pools[b])}/{n_docs} docs")
        pools[b] = pools[b][:n_docs]
        with open(bucket_path(b), "w", encoding="utf-8") as f:
            for r in pools[b]:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    return pools

raw_pools = select_c4_buckets(cfg.buckets, cfg.raw_pool_size)

for b, rows in raw_pools.items():
    assert all(int(x["sha256"][:8], 16) % cfg.c4_modulus == b for x in rows)
    print("bucket", b, "first hashes:", [x["sha256"][:10] for x in rows[:3]])

clear_dataset_cache()
disk_status("After C4 selection")


In [ ]:
# ============================================================
# GENERIC HELPERS
# ============================================================
def sanitize(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)


def get_by_path(obj, path):
    cur = obj
    for part in path.split("."):
        cur = getattr(cur, part)
    return cur


def resolve_layers(model):
    for path in [
        "model.layers",
        "model.model.layers",
        "model.language_model.layers",
        "model.language_model.model.layers",
        "transformer.h",
    ]:
        try:
            layers = get_by_path(model, path)
            if len(layers) > 0:
                return layers, path
        except Exception:
            pass
    raise RuntimeError("Could not resolve transformer block list")


def resolve_mlp(block):
    for name in ["mlp", "feed_forward", "ffn", "feed_forward_layer"]:
        if hasattr(block, name):
            return getattr(block, name), name
    for name, module in block.named_modules():
        lname = name.lower()
        if name and any(x in lname for x in ["mlp", "feed_forward", "ffn"]):
            return module, name
    raise RuntimeError("Could not resolve MLP module")


def model_input_device(model):
    for path in ["model.embed_tokens", "model.model.embed_tokens", "transformer.wte"]:
        try:
            return next(get_by_path(model, path).parameters()).device
        except Exception:
            pass
    return next(model.parameters()).device


def unwrap_first_tensor(x):
    if torch.is_tensor(x):
        return x
    if isinstance(x, (tuple, list)):
        for y in x:
            if torch.is_tensor(y):
                return y
    raise TypeError(f"No tensor found in output type {type(x)}")


def replace_first_tensor(container, new_tensor):
    if torch.is_tensor(container):
        return new_tensor
    if isinstance(container, tuple):
        out = list(container)
        for i, x in enumerate(out):
            if torch.is_tensor(x):
                out[i] = new_tensor
                return tuple(out)
    if isinstance(container, list):
        out = list(container)
        for i, x in enumerate(out):
            if torch.is_tensor(x):
                out[i] = new_tensor
                return out
    raise TypeError(f"Could not replace first tensor in {type(container)}")


def tokenize_pool(tokenizer, raw_rows, n, n_tokens):
    toks, hashes = [], []
    for row in raw_rows:
        ids = tokenizer(row["text"], add_special_tokens=False, truncation=False)["input_ids"]
        if len(ids) < n_tokens:
            continue
        toks.append(torch.tensor(ids[:n_tokens], dtype=torch.long))
        hashes.append(row["sha256"])
        if len(toks) >= n:
            break
    if len(toks) < n:
        raise RuntimeError(f"Only {len(toks)} docs tokenize to >= {n_tokens} tokens; need {n}")
    return toks, hashes


def spearman(x, y):
    r = spearmanr(np.asarray(x), np.asarray(y), nan_policy="omit")
    return float(r.statistic), float(r.pvalue)


def native_to_patch_metrics(z_native, z_patch):
    """z_native [V], z_patch [B,V]. Returns several exact distribution-change outcomes."""
    logp = F.log_softmax(z_native.float(), dim=-1)
    p = logp.exp()
    logq = F.log_softmax(z_patch.float(), dim=-1)
    q = logq.exp()

    kl_pq = torch.sum(p.unsqueeze(0) * (logp.unsqueeze(0) - logq), dim=-1)
    kl_qp = torch.sum(q * (logq - logp.unsqueeze(0)), dim=-1)

    m = 0.5 * (p.unsqueeze(0) + q)
    logm = torch.log(torch.clamp(m, min=1e-30))
    js = 0.5 * torch.sum(p.unsqueeze(0) * (logp.unsqueeze(0) - logm), dim=-1)
    js += 0.5 * torch.sum(q * (logq - logm), dim=-1)

    return {
        "kl": torch.clamp(kl_pq, min=0.0),
        "reverse_kl": torch.clamp(kl_qp, min=0.0),
        "js": torch.clamp(js, min=0.0),
    }


def fd_sensitivity_metrics(z_native, z_plus, z_minus, eps, return_dz=False):
    """All downstream baselines use the exact same centered-FD directional logit derivative."""
    dz = (z_plus.float() - z_minus.float()) / (2.0 * eps)
    p = F.softmax(z_native.float(), dim=-1).unsqueeze(0)

    p_mean = torch.sum(p * dz, dim=-1)
    p_second = torch.sum(p * dz.square(), dim=-1)
    q_fisher = 0.5 * torch.clamp(p_second - p_mean.square(), min=0.0)
    q_p2 = 0.5 * p_second

    raw_mean = dz.mean(dim=-1)
    q_l2 = 0.5 * dz.square().mean(dim=-1)
    q_l2_centered = 0.5 * (dz - raw_mean.unsqueeze(-1)).square().mean(dim=-1)

    out = {
        "q_fisher": q_fisher,
        "q_p2": q_p2,
        "q_l2": q_l2,
        "q_l2_centered": q_l2_centered,
    }
    if return_dz:
        out["dz"] = dz
    return out


def efficient_two_way_residual(df, value_col, max_iter=100, tol=1e-12):
    """Projection onto residuals of Y_ij = mu + alpha_i + beta_j + eps_ij."""
    y = df[value_col].astype(float).to_numpy().copy()
    h_raw = df["host"].astype(int).to_numpy()
    d_raw = df["donor"].astype(int).to_numpy()
    _, h = np.unique(h_raw, return_inverse=True)
    _, d = np.unique(d_raw, return_inverse=True)
    nh, nd = h.max()+1, d.max()+1
    hc = np.bincount(h, minlength=nh).astype(float)
    dc = np.bincount(d, minlength=nd).astype(float)
    res = y - np.mean(y)

    for _ in range(max_iter):
        hm = np.bincount(h, weights=res, minlength=nh) / np.maximum(hc, 1)
        res -= hm[h]
        dm = np.bincount(d, weights=res, minlength=nd) / np.maximum(dc, 1)
        res -= dm[d]
        if max(np.max(np.abs(hm)), np.max(np.abs(dm))) < tol:
            break
    return res


def standardized_rank(x):
    r = rankdata(np.asarray(x, dtype=float), method="average")
    r = r - r.mean()
    sd = r.std()
    return r / sd if sd > 0 else r


In [ ]:
# ============================================================
# TARGET CONTEXT — CAPTURE / PATCH / FD / EXACT JVP
# ============================================================
class TargetContext:
    def __init__(self, model, tokenizer, token_ids, layer_idx, pair_batch):
        self.model = model
        self.tokenizer = tokenizer
        self.ids = token_ids
        self.layer_idx = layer_idx
        self.pair_batch = pair_batch
        self.layers, self.layers_path = resolve_layers(model)
        self.block = self.layers[layer_idx]
        self.module, self.module_name = resolve_mlp(self.block)
        self.input_device = model_input_device(model)
        self.module_device = next(self.module.parameters()).device
        self.patch_pos = cfg.core_tokens - 1
        self.M = None
        self.U = None
        self.Z0 = None
        self.M_pca = None

    def capture_one(self, ids_1d):
        cap = {}

        def pre_hook(module, inputs):
            x = unwrap_first_tensor(inputs)
            cap["m"] = x[0, self.patch_pos, :].detach().float().cpu()

        def out_hook(module, inputs, output):
            y = unwrap_first_tensor(output)
            cap["u"] = y[0, self.patch_pos, :].detach().float().cpu()

        h1 = self.module.register_forward_pre_hook(pre_hook)
        h2 = self.module.register_forward_hook(out_hook)
        try:
            out = self.model(
                input_ids=ids_1d.unsqueeze(0).to(self.input_device),
                use_cache=False,
                return_dict=True,
            )
            z = out.logits[0, self.patch_pos, :].detach().float().cpu()
        finally:
            h1.remove(); h2.remove()
        return cap["m"], cap["u"], z

    def capture_native(self, cache_dir):
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)
        tensor_path = cache_dir / "native_cache.pt"
        if cfg.resume and tensor_path.exists():
            d = torch.load(tensor_path, map_location="cpu")
            self.M, self.U, self.Z0, self.M_pca = d["M"], d["U"], d["Z0"], d["M_pca"]
            print("RESUME: loaded native cache", tensor_path)
            return

        ms, us, zs = [], [], []
        for ids in tqdm(self.ids, desc=f"Native MLP L{self.layer_idx}"):
            m, u, z = self.capture_one(ids)
            ms.append(m); us.append(u); zs.append(z)
        self.M = torch.stack(ms)
        self.U = torch.stack(us)
        self.Z0 = torch.stack(zs)

        pca_dim = min(32, len(self.ids)-1, self.M.shape[1])
        pca = PCA(n_components=pca_dim, random_state=cfg.seed)
        self.M_pca = torch.tensor(pca.fit_transform(self.M.numpy()), dtype=torch.float32)
        torch.save({"M": self.M, "U": self.U, "Z0": self.Z0, "M_pca": self.M_pca}, tensor_path)

    def patched_logits(self, receiver, replacement_batch):
        ids = self.ids[receiver]
        B = replacement_batch.shape[0]
        batch_ids = ids.unsqueeze(0).repeat(B, 1).to(self.input_device)

        def hook(module, inputs, output):
            y = unwrap_first_tensor(output)
            rep = replacement_batch.to(device=y.device, dtype=y.dtype)
            y2 = y.clone()
            y2[:, self.patch_pos, :] = rep
            return replace_first_tensor(output, y2)

        h = self.module.register_forward_hook(hook)
        try:
            out = self.model(input_ids=batch_ids, use_cache=False, return_dict=True)
            return out.logits[:, self.patch_pos, :].detach().float().cpu()
        finally:
            h.remove()

    def fd_metrics(self, receiver, delta_batch, eps, return_dz=False):
        u = self.U[receiver].unsqueeze(0)
        zp = self.patched_logits(receiver, u + eps * delta_batch)
        zm = self.patched_logits(receiver, u - eps * delta_batch)
        return fd_sensitivity_metrics(self.Z0[receiver], zp, zm, eps, return_dz=return_dz)

    def source_metrics(self, host, donors):
        js = torch.tensor(donors, dtype=torch.long)
        mi = self.M[host].unsqueeze(0)
        ui = self.U[host].unsqueeze(0)
        mj = self.M[js]
        uj = self.U[js]
        dm = mj - mi
        du = uj - ui
        return {
            "dm_norm": torch.linalg.norm(dm, dim=-1),
            "du_norm": torch.linalg.norm(du, dim=-1),
            "state_dist_pca": torch.linalg.norm(self.M_pca[js] - self.M_pca[host].unsqueeze(0), dim=-1),
        }

    def self_patch_check(self):
        z = self.patched_logits(0, self.U[0:1])
        k = float(native_to_patch_metrics(self.Z0[0], z)["kl"][0])
        print("Self-patch KL:", k)
        if k >= 5e-4:
            raise RuntimeError(f"Self-patch KL too large: {k}")
        return k


In [ ]:
# ============================================================
# CORE 256 x 255 MATRIX — RESUMABLE BY HOST
# ============================================================
def run_core_matrix(ctx, model_dir):
    model_dir = Path(model_dir)
    chunk_dir = model_dir / "core_host_chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)
    n = len(ctx.ids)

    for i in tqdm(range(n), desc="Core hosts"):
        chunk_path = chunk_dir / f"host_{i:03d}.csv"
        if cfg.resume and chunk_path.exists():
            continue

        rows = []
        donors = [j for j in range(n) if j != i]
        for s in range(0, len(donors), ctx.pair_batch):
            js = donors[s:s+ctx.pair_batch]
            jt = torch.tensor(js, dtype=torch.long)
            delta = ctx.U[jt] - ctx.U[i].unsqueeze(0)

            zfull = ctx.patched_logits(i, ctx.U[jt])
            outcomes = native_to_patch_metrics(ctx.Z0[i], zfull)
            fd = ctx.fd_metrics(i, delta, cfg.stable_eps)
            src = ctx.source_metrics(i, js)

            for b, j in enumerate(js):
                rows.append({
                    "host": i,
                    "donor": j,
                    "kl": float(outcomes["kl"][b]),
                    "reverse_kl": float(outcomes["reverse_kl"][b]),
                    "js": float(outcomes["js"][b]),
                    "q_fisher": float(fd["q_fisher"][b]),
                    "q_p2": float(fd["q_p2"][b]),
                    "q_l2": float(fd["q_l2"][b]),
                    "q_l2_centered": float(fd["q_l2_centered"][b]),
                    "du_norm": float(src["du_norm"][b]),
                    "dm_norm": float(src["dm_norm"][b]),
                    "state_dist_pca": float(src["state_dist_pca"][b]),
                })
        pd.DataFrame(rows).to_csv(chunk_path, index=False)

        if (i + 1) % 16 == 0:
            gc.collect(); torch.cuda.empty_cache()

    df = pd.concat([pd.read_csv(p) for p in sorted(chunk_dir.glob("host_*.csv"))], ignore_index=True)
    expected = n * (n - 1)
    if len(df) != expected:
        raise RuntimeError(f"Core matrix has {len(df)} rows, expected {expected}")
    df.to_csv(model_dir / "core_pairs.csv", index=False)
    if cfg.cleanup_completed_chunks:
        shutil.rmtree(chunk_dir, ignore_errors=True)
    return df


In [ ]:
# ============================================================
# CORE ANALYSES — FISHER VS SIMPLE DOWNSTREAM BASELINES
# ============================================================
PREDICTORS = ["q_fisher", "q_p2", "q_l2_centered", "q_l2", "du_norm", "state_dist_pca"]


def joint_context_permutation(df, pred_col, reps, seed):
    """Jointly relabel both host and donor context axes of predictor matrix."""
    n = max(df.host.max(), df.donor.max()) + 1
    k_mat = np.full((n,n), np.nan)
    q_mat = np.full((n,n), np.nan)
    for r in df[["host","donor","kl",pred_col]].itertuples(index=False):
        k_mat[int(r.host), int(r.donor)] = float(r.kl)
        q_mat[int(r.host), int(r.donor)] = float(getattr(r, pred_col))

    mask = ~np.eye(n, dtype=bool)
    k_rank = standardized_rank(k_mat[mask])
    q_rank = rankdata(q_mat[mask], method="average")
    q_rank_mat = np.full((n,n), np.nan)
    q_rank_mat[mask] = q_rank
    obs_q = standardized_rank(q_mat[mask])
    obs = float(np.mean(k_rank * obs_q))

    rng = np.random.default_rng(seed)
    null = np.empty(reps, dtype=float)
    for b in range(reps):
        perm = rng.permutation(n)
        qperm = q_rank_mat[np.ix_(perm, perm)][mask]
        qperm = (qperm - np.nanmean(qperm)) / np.nanstd(qperm)
        null[b] = float(np.mean(k_rank * qperm))
    p = (1 + np.sum(np.abs(null) >= abs(obs))) / (reps + 1)
    return obs, float(p), null


def within_group_rhos(df, group_col, pred_col, outcome_col="kl"):
    rows = []
    for g, x in df.groupby(group_col):
        if len(x) < 4:
            continue
        rho, _ = spearman(x[pred_col], x[outcome_col])
        rows.append({group_col: g, "rho": rho})
    return pd.DataFrame(rows)


def donor_selection_compare(df, bank_sizes=(2,4,8,16,32)):
    rows = []
    for M in bank_sizes:
        for host, g in df.groupby("host"):
            bank = g.nsmallest(M, "state_dist_pca")
            if len(bank) < M:
                continue
            for selector in ["q_fisher", "q_l2_centered", "q_l2", "state_dist_pca"]:
                chosen = bank.loc[bank[selector].idxmin()]
                rows.append({
                    "M": M,
                    "host": int(host),
                    "selector": selector,
                    "selected_kl": float(chosen.kl),
                    "selected_donor": int(chosen.donor),
                })
    return pd.DataFrame(rows)


def analyze_core(df, model_dir):
    model_dir = Path(model_dir)
    result_rows = []
    df2 = df.copy()
    df2["log_kl"] = np.log1p(df2["kl"])

    for pred in PREDICTORS:
        df2[f"log_{pred}"] = np.log1p(np.maximum(df2[pred].astype(float), 0))
        raw_rho, raw_p = spearman(df2[pred], df2["kl"])
        kres = efficient_two_way_residual(df2, "log_kl")
        qres = efficient_two_way_residual(df2, f"log_{pred}")
        resid_rho, resid_p = spearman(qres, kres)

        wh = within_group_rhos(df2, "host", pred)
        wd = within_group_rhos(df2, "donor", pred)
        result_rows.append({
            "predictor": pred,
            "raw_rho": raw_rho,
            "raw_p": raw_p,
            "two_way_resid_rho": resid_rho,
            "two_way_resid_p": resid_p,
            "median_within_host_rho": float(wh.rho.median()),
            "positive_host_fraction": float((wh.rho > 0).mean()),
            "median_within_donor_rho": float(wd.rho.median()),
            "positive_donor_fraction": float((wd.rho > 0).mean()),
        })

    summary = pd.DataFrame(result_rows).sort_values("two_way_resid_rho", ascending=False)
    summary.to_csv(model_dir / "core_predictor_comparison.csv", index=False)

    # Robustness of outcome metric, no extra model passes.
    out_rows = []
    for outcome in ["kl", "reverse_kl", "js"]:
        for pred in ["q_fisher", "q_l2_centered", "q_l2"]:
            rho, p = spearman(df2[pred], df2[outcome])
            out_rows.append({"outcome": outcome, "predictor": pred, "rho": rho, "p": p})
    pd.DataFrame(out_rows).to_csv(model_dir / "outcome_robustness.csv", index=False)

    # Shared-context joint permutation for Fisher and best simple downstream baseline.
    simple_cols = ["q_p2", "q_l2_centered", "q_l2"]
    best_simple = summary[summary.predictor.isin(simple_cols)].iloc[0].predictor
    perm_rows = []
    for pred in ["q_fisher", best_simple]:
        obs, p, null = joint_context_permutation(df2, pred, cfg.permutation_reps, cfg.seed + 100)
        perm_rows.append({
            "predictor": pred,
            "observed_raw_spearman": obs,
            "permutation_p": p,
            "null_abs_max": float(np.max(np.abs(null))),
        })
    pd.DataFrame(perm_rows).to_csv(model_dir / "joint_context_permutation.csv", index=False)

    sel = donor_selection_compare(df2)
    sel.to_csv(model_dir / "downstream_selection_comparison.csv", index=False)
    sel_sum = sel.groupby(["M","selector"])["selected_kl"].median().reset_index(name="median_selected_kl")
    sel_sum.to_csv(model_dir / "downstream_selection_summary.csv", index=False)

    return summary, best_simple, sel


In [ ]:
# ============================================================
# FIXED-DELTA DESIGN + PHYSICAL CROSSED-RECEIVER TEST
# ============================================================
def design_pairs(n_contexts):
    return pd.DataFrame(
        [(i, j) for i in range(n_contexts) for j in range(n_contexts) if j != i],
        columns=["host", "donor"]
    )

def make_crossed_design(n_contexts, bucket, receivers_per_delta, design_dir):
    design_dir = Path(design_dir)
    design_dir.mkdir(parents=True, exist_ok=True)
    design_path = design_dir / f"crossed_design_bucket{bucket}_r{receivers_per_delta}.json"

    if cfg.resume and design_path.exists():
        return json.loads(design_path.read_text())

    pairs = design_pairs(n_contexts)
    chosen = pairs.sample(
        n=min(cfg.n_cross_deltas, len(pairs)),
        random_state=cfg.seed + 201,
    )

    rng = np.random.default_rng(cfg.seed + 200)
    design = []
    for delta_id, row in enumerate(chosen.itertuples(index=False)):
        i, j = int(row.host), int(row.donor)
        pool = np.array([k for k in range(n_contexts) if k not in (i, j)], dtype=int)
        n_alt = min(max(receivers_per_delta - 1, 0), len(pool))
        alt = rng.choice(pool, size=n_alt, replace=False) if n_alt else np.array([], dtype=int)
        design.append({
            "delta_id": int(delta_id),
            "origin_host": i,
            "origin_donor": j,
            "receivers": [i] + [int(x) for x in alt],
        })

    design_path.write_text(json.dumps(design, indent=2))
    return design

def run_crossed_receiver_test(ctx, model_dir, bucket, receivers_per_delta):
    model_dir = Path(model_dir)
    final_path = model_dir / "cross_receiver_physical.csv"
    if cfg.resume and final_path.exists():
        print("RESUME:", final_path)
        return pd.read_csv(final_path)

    chunk_dir = model_dir / "cross_receiver_chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)

    design = make_crossed_design(
        len(ctx.ids),
        bucket=bucket,
        receivers_per_delta=receivers_per_delta,
        design_dir=model_dir,
    )

    tasks_by_receiver = defaultdict(list)
    for item in design:
        i, j, did = item["origin_host"], item["origin_donor"], item["delta_id"]
        delta = ctx.U[j] - ctx.U[i]
        for k in item["receivers"]:
            tasks_by_receiver[int(k)].append((did, i, j, delta))

    for k in tqdm(sorted(tasks_by_receiver), desc=f"Physical receivers b{bucket} L{ctx.layer_idx}"):
        chunk_path = chunk_dir / f"receiver_{k:03d}.csv"
        if cfg.resume and chunk_path.exists():
            continue

        rows = []
        tasks = tasks_by_receiver[k]
        for s in range(0, len(tasks), ctx.pair_batch):
            bt = tasks[s:s+ctx.pair_batch]
            deltas = torch.stack([x[3] for x in bt])
            replacements = ctx.U[k].unsqueeze(0) + deltas

            zfull = ctx.patched_logits(k, replacements)
            outcomes = native_to_patch_metrics(ctx.Z0[k], zfull)
            fd = ctx.fd_metrics(k, deltas, cfg.stable_eps)

            for b, (did, i, j, delta) in enumerate(bt):
                rows.append({
                    "delta_id": int(did),
                    "origin_host": int(i),
                    "origin_donor": int(j),
                    "receiver": int(k),
                    "delta_norm": float(torch.linalg.norm(delta)),
                    "kl_cross": float(outcomes["kl"][b]),
                    "reverse_kl_cross": float(outcomes["reverse_kl"][b]),
                    "js_cross": float(outcomes["js"][b]),
                    "q_fisher": float(fd["q_fisher"][b]),
                    "q_p2": float(fd["q_p2"][b]),
                    "q_l2": float(fd["q_l2"][b]),
                    "q_l2_centered": float(fd["q_l2_centered"][b]),
                })

        pd.DataFrame(rows).to_csv(chunk_path, index=False)

    cross = pd.concat(
        [pd.read_csv(p) for p in sorted(chunk_dir.glob("receiver_*.csv"))],
        ignore_index=True
    )

    expected = cfg.n_cross_deltas * receivers_per_delta
    if len(cross) != expected:
        raise RuntimeError(f"Cross rows {len(cross)} != expected {expected}")

    # Original-host anchor: nonfatal robust QA audit.
    anchor = cross[cross.receiver == cross.origin_host].copy()
    if len(anchor):
        anchor_errors = []
        for r in anchor.itertuples(index=False):
            i, j = int(r.origin_host), int(r.origin_donor)
            expected_kl = float(
                native_to_patch_metrics(
                    ctx.Z0[i],
                    ctx.patched_logits(i, ctx.U[j:j+1])
                )["kl"][0]
            )
            anchor_errors.append(abs(float(r.kl_cross) - expected_kl))

        anchor["abs_kl_error"] = anchor_errors
        anchor.to_csv(model_dir / "cross_receiver_anchor_audit.csv", index=False)

        q = anchor.abs_kl_error.quantile([0.5, 0.9, 0.95, 0.99, 1.0])
        print("Anchor error quantiles:")
        print(q)
        if float(q.loc[0.95]) > 1e-4 or float(q.loc[0.99]) > 1e-3:
            print("WARNING: anchor tail is larger than expected; inspect before paper use.")

    cross.to_csv(final_path, index=False)

    if cfg.cleanup_completed_chunks:
        shutil.rmtree(chunk_dir, ignore_errors=True)

    return cross

def analyze_crossed_receiver(cross, model_dir):
    model_dir = Path(model_dir)
    rows = []
    pred_cols = ["q_fisher", "q_p2", "q_l2_centered", "q_l2"]

    for pred in pred_cols:
        pooled_rho, pooled_p = spearman(cross[pred], cross["kl_cross"])

        per_delta = []
        for did, g in cross.groupby("delta_id"):
            rho_all, _ = spearman(g[pred], g["kl_cross"])
            g_alt = g[g.receiver != g.origin_host]
            rho_alt, _ = spearman(g_alt[pred], g_alt["kl_cross"]) if len(g_alt) >= 4 else (float("nan"), float("nan"))
            per_delta.append({
                "delta_id": int(did),
                "rho_all": rho_all,
                "rho_alt_only": rho_alt,
            })

        pr = pd.DataFrame(per_delta)
        vals_all = pr.rho_all.replace([np.inf, -np.inf], np.nan).dropna()
        vals_alt = pr.rho_alt_only.replace([np.inf, -np.inf], np.nan).dropna()

        try:
            wp = float(wilcoxon(vals_alt, alternative="greater").pvalue)
        except Exception:
            wp = float("nan")

        rows.append({
            "predictor": pred,
            "pooled_rho": pooled_rho,
            "pooled_p": pooled_p,
            "median_within_fixed_delta_rho_all": float(vals_all.median()),
            "median_within_alt_receivers_rho": float(vals_alt.median()),
            "mean_within_alt_receivers_rho": float(vals_alt.mean()),
            "positive_delta_fraction_alt": float((vals_alt > 0).mean()),
            "wilcoxon_alt_greater_p": wp,
            "n_deltas": int(len(vals_alt)),
        })

        pr.to_csv(model_dir / f"cross_receiver_per_delta_{pred}.csv", index=False)

    summary = pd.DataFrame(rows).sort_values("median_within_alt_receivers_rho", ascending=False)
    summary.to_csv(model_dir / "cross_receiver_predictor_summary.csv", index=False)

    # Stable effect-spread summaries, avoiding huge max/min ratios from near-zero KL.
    var_rows = []
    for did, g in cross.groupby("delta_id"):
        v = g[g.receiver != g.origin_host].kl_cross.to_numpy(float)
        logv = np.log10(np.maximum(v, 1e-12))
        var_rows.append({
            "delta_id": int(did),
            "alt_min_kl": float(np.min(v)),
            "alt_median_kl": float(np.median(v)),
            "alt_max_kl": float(np.max(v)),
            "alt_log10_kl_iqr": float(np.quantile(logv, 0.75) - np.quantile(logv, 0.25)),
            "alt_log10_kl_std": float(np.std(logv)),
        })

    variation = pd.DataFrame(var_rows)
    variation.to_csv(model_dir / "cross_receiver_same_delta_variation.csv", index=False)
    return summary, variation


In [ ]:

# ============================================================
# STORAGE-SAFE TWO-T4 MODEL LOADER — V3 FIX
# ============================================================
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

def load_model(spec):
    """
    Load a 7–8B model across BOTH T4s.

    Important V3 fix:
    - use device_map="balanced" (NOT balanced_low_0)
    - use 12000 MiB ceiling on each GPU
    - do not clear the HF model cache immediately before every retry
    - reject CPU/disk offload
    - reject highly imbalanced loads before long experiments
    """
    gc.collect()
    torch.cuda.empty_cache()

    MODEL_CACHE.mkdir(parents=True, exist_ok=True)

    # Require a clean CUDA state before loading.
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 2**30
        if alloc > 0.75:
            raise RuntimeError(
                f"GPU {i} already has {alloc:.2f} GiB allocated before model load. "
                "Restart the Kaggle kernel/session before retrying."
            )

    free_disk = disk_status("Before model download")
    # Keep enough working-disk headroom for outputs/checkpoints.
    if free_disk < 4.0:
        raise RuntimeError(
            f"Only {free_disk:.2f} GiB free in /kaggle/working. "
            "Delete stale HF/model caches or start a fresh Kaggle session."
        )

    tok = AutoTokenizer.from_pretrained(
        spec["model_id"],
        token=HF_TOKEN,
        trust_remote_code=True,
        cache_dir=str(MODEL_CACHE),
    )
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    max_memory = {
        0: "12000MiB",
        1: "12000MiB",
    }

    print(
        "\nLoading:", spec["model_id"],
        "\ndevice_map = balanced",
        "\nmax_memory =", max_memory
    )

    model = AutoModelForCausalLM.from_pretrained(
        spec["model_id"],
        token=HF_TOKEN,
        trust_remote_code=True,
        dtype=torch.float16,
        device_map="balanced",
        max_memory=max_memory,
        low_cpu_mem_usage=True,
        cache_dir=str(MODEL_CACHE),
    )

    model.eval()
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    for p in model.parameters():
        p.requires_grad_(False)

    dmap = getattr(model, "hf_device_map", {})

    print("\nDEVICE MAP:")
    for k, v in dmap.items():
        print(k, "->", v)

    bad = {
        k: v for k, v in dmap.items()
        if str(v).lower() in {"cpu", "disk"}
    }
    if bad:
        # Delete the local Python reference before raising so a retry
        # does not leave a large live model around.
        del model
        gc.collect()
        torch.cuda.empty_cache()
        raise RuntimeError(
            "CPU/disk offload detected:\n"
            + "\n".join(f"{k} -> {v}" for k, v in bad.items())
            + "\nRestart the kernel before trying a different loader."
        )

    gpu_alloc = []
    print("\nCUDA after load:")
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        alloc = torch.cuda.memory_allocated(i) / 2**30
        gpu_alloc.append(alloc)
        print(
            f"GPU {i}: "
            f"allocated={alloc:.2f} GiB | "
            f"reserved={torch.cuda.memory_reserved(i)/2**30:.2f} GiB | "
            f"free={free/2**30:.2f}/{total/2**30:.2f} GiB"
        )

    # A long run should not begin if one T4 is nearly full.
    if max(gpu_alloc) > 10.5:
        del model
        gc.collect()
        torch.cuda.empty_cache()
        raise RuntimeError(
            f"Imbalanced load: max GPU allocation is {max(gpu_alloc):.2f} GiB. "
            "Restart the kernel and inspect the device map before running Phase 1."
        )

    return model, tok

def unload_model(model=None, tokenizer=None, ctx=None, delete_model_cache=True):
    """
    Release CUDA state. Optionally delete the HF model cache when switching
    model families to keep Kaggle storage under control.
    """
    try:
        if ctx is not None:
            del ctx
    except Exception:
        pass

    try:
        if model is not None:
            del model
    except Exception:
        pass

    try:
        if tokenizer is not None:
            del tokenizer
    except Exception:
        pass

    gc.collect()
    torch.cuda.empty_cache()

    if delete_model_cache:
        clear_model_cache()

    gpu_status("After model unload")
    disk_status("After model unload/cache cleanup")


In [ ]:
# ============================================================
# STAGE HELPERS
# ============================================================
stage_status = []

def resolve_layer_idx(model, rel_depth):
    layers, path = resolve_layers(model)
    idx = int(round(rel_depth * (len(layers) - 1)))
    return idx, len(layers), path

def stage_dir(model_key, bucket, layer_idx, kind):
    return OUT_ROOT / "stages" / model_key / f"bucket{bucket}" / f"layer{layer_idx:02d}" / kind

def record_stage(row):
    stage_status.append(row)
    pd.DataFrame(stage_status).to_csv(OUT_ROOT / "stage_status.csv", index=False)

def run_cross_stage(model, tokenizer, model_key, bucket, rel_depth, receivers_per_delta):
    layer_idx, n_layers, layer_path = resolve_layer_idx(model, rel_depth)
    sdir = stage_dir(model_key, bucket, layer_idx, "cross")
    sdir.mkdir(parents=True, exist_ok=True)

    summary_path = sdir / "cross_receiver_predictor_summary.csv"
    if cfg.resume and summary_path.exists():
        print("RESUME completed cross stage:", sdir)
        return pd.read_csv(summary_path)

    token_ids, hashes = tokenize_pool(tokenizer, raw_pools[bucket], cfg.n_contexts, cfg.core_tokens)
    (sdir / "document_hashes.json").write_text(json.dumps(hashes, indent=2))

    ctx = TargetContext(model, tokenizer, token_ids, layer_idx, cfg.pair_batch)
    ctx.capture_native(sdir)
    self_kl = ctx.self_patch_check()

    t0 = time.time()
    cross = run_crossed_receiver_test(ctx, sdir, bucket, receivers_per_delta)
    cross_summary, variation = analyze_crossed_receiver(cross, sdir)
    minutes = (time.time() - t0) / 60

    fish = cross_summary[cross_summary.predictor == "q_fisher"].iloc[0]
    best_simple = (
        cross_summary[cross_summary.predictor.isin(["q_p2", "q_l2_centered", "q_l2"])]
        .sort_values("median_within_alt_receivers_rho", ascending=False)
        .iloc[0]
    )

    metadata = {
        "kind": "cross",
        "model_key": model_key,
        "model_id": MODEL_SPECS[model_key]["model_id"],
        "bucket": int(bucket),
        "layer_idx": int(layer_idx),
        "n_layers": int(n_layers),
        "relative_depth": float(layer_idx / max(n_layers - 1, 1)),
        "receivers_per_delta": int(receivers_per_delta),
        "n_interventions": int(len(cross)),
        "self_patch_kl": float(self_kl),
        "fisher_pooled_rho": float(fish.pooled_rho),
        "fisher_median_alt_rho": float(fish.median_within_alt_receivers_rho),
        "fisher_positive_delta_fraction": float(fish.positive_delta_fraction_alt),
        "best_simple": str(best_simple.predictor),
        "best_simple_median_alt_rho": float(best_simple.median_within_alt_receivers_rho),
        "fisher_minus_best_simple_alt_rho": float(
            fish.median_within_alt_receivers_rho - best_simple.median_within_alt_receivers_rho
        ),
        "minutes": float(minutes),
    }
    (sdir / "stage_summary.json").write_text(json.dumps(metadata, indent=2))
    record_stage(metadata)
    print(json.dumps(metadata, indent=2))
    del ctx, cross
    gc.collect()
    torch.cuda.empty_cache()
    return cross_summary

def run_full_core_stage(model, tokenizer, model_key, bucket, rel_depth):
    layer_idx, n_layers, layer_path = resolve_layer_idx(model, rel_depth)
    sdir = stage_dir(model_key, bucket, layer_idx, "full_core")
    sdir.mkdir(parents=True, exist_ok=True)

    summary_path = sdir / "core_predictor_comparison.csv"
    if cfg.resume and summary_path.exists():
        print("RESUME completed full-core stage:", sdir)
        return pd.read_csv(summary_path)

    token_ids, hashes = tokenize_pool(tokenizer, raw_pools[bucket], cfg.n_contexts, cfg.core_tokens)
    (sdir / "document_hashes.json").write_text(json.dumps(hashes, indent=2))

    ctx = TargetContext(model, tokenizer, token_ids, layer_idx, cfg.pair_batch)
    ctx.capture_native(sdir)
    self_kl = ctx.self_patch_check()

    t0 = time.time()
    core = run_core_matrix(ctx, sdir)
    core_summary, best_simple, selection = analyze_core(core, sdir)
    minutes = (time.time() - t0) / 60

    fish = core_summary[core_summary.predictor == "q_fisher"].iloc[0]
    simple = core_summary[core_summary.predictor == best_simple].iloc[0]

    metadata = {
        "kind": "full_core",
        "model_key": model_key,
        "model_id": MODEL_SPECS[model_key]["model_id"],
        "bucket": int(bucket),
        "layer_idx": int(layer_idx),
        "n_layers": int(n_layers),
        "relative_depth": float(layer_idx / max(n_layers - 1, 1)),
        "n_pairs": int(len(core)),
        "self_patch_kl": float(self_kl),
        "fisher_raw_rho": float(fish.raw_rho),
        "fisher_two_way_resid_rho": float(fish.two_way_resid_rho),
        "best_simple": str(best_simple),
        "best_simple_two_way_resid_rho": float(simple.two_way_resid_rho),
        "fisher_minus_best_simple_resid_rho": float(
            fish.two_way_resid_rho - simple.two_way_resid_rho
        ),
        "minutes": float(minutes),
    }
    (sdir / "stage_summary.json").write_text(json.dumps(metadata, indent=2))
    record_stage(metadata)
    print(json.dumps(metadata, indent=2))
    del ctx, core
    gc.collect()
    torch.cuda.empty_cache()
    return core_summary


## Phase 1 — REQUIRED: Mistral primary-depth full replication

This is the highest-priority phase.

Expected wall time: roughly **4–5 hours**.


In [ ]:

# ============================================================
# MISTRAL LOAD-ONLY SMOKE TEST
# Run this before the long Phase 1 experiment.
# ============================================================
spec = MODEL_SPECS["mistral"]

# If this cell raises, RESTART THE KERNEL before retrying.
model, tokenizer = load_model(spec)

print("\nMISTRAL LOAD TEST PASSED.")
print("Keep this model loaded and run the NEXT cell.")


In [ ]:

# ============================================================
# PHASE 1 — MISTRAL PRIMARY FULL CORE + PHYSICAL CROSSING
# Reuses the model from the load-only smoke test above.
# ============================================================
assert model is not None and tokenizer is not None, \
    "Run the Mistral load-only smoke test cell first."

try:
    # Full pair matrix at primary depth.
    run_full_core_stage(
        model, tokenizer,
        model_key="mistral",
        bucket=1,
        rel_depth=0.40,
    )

    # Direct physical crossing at the same depth.
    run_cross_stage(
        model, tokenizer,
        model_key="mistral",
        bucket=1,
        rel_depth=0.40,
        receivers_per_delta=cfg.receivers_bucket1,
    )

    write_checkpoint_zip()
    print("\nPHASE 1 DONE. Download checkpoint now if Kaggle is unstable.")

finally:
    unload_model(model, tokenizer, delete_model_cache=True)
    model = None
    tokenizer = None


## Phase 2 — REQUIRED: physical receiver dependence across depth

Runs bucket 1 at four depths for both models. The Mistral 40% stage automatically resumes/skips from Phase 1. Qwen does **not** rerun its 65,280-pair matrix.


In [ ]:
# ============================================================
# PHASE 2A — MISTRAL DEPTH REPLICATION
# ============================================================
if phase_ok(2.0, "Mistral depth replication"):
    spec = MODEL_SPECS["mistral"]
    model = tokenizer = None
    try:
        model, tokenizer = load_model(spec)
        for rel in cfg.rel_depths:
            run_cross_stage(
                model, tokenizer,
                model_key="mistral",
                bucket=1,
                rel_depth=float(rel),
                receivers_per_delta=cfg.receivers_bucket1,
            )
    finally:
        unload_model(model, tokenizer)
    write_checkpoint_zip()


In [ ]:
# ============================================================
# PHASE 2B — QWEN DEPTH REPLICATION
# No Qwen 65k core rerun.
# ============================================================
if phase_ok(2.5, "Qwen depth replication"):
    spec = MODEL_SPECS["qwen"]
    model = tokenizer = None
    try:
        model, tokenizer = load_model(spec)
        for rel in cfg.rel_depths:
            run_cross_stage(
                model, tokenizer,
                model_key="qwen",
                bucket=1,
                rel_depth=float(rel),
                receivers_per_delta=cfg.receivers_bucket1,
            )
    finally:
        unload_model(model, tokenizer)
    write_checkpoint_zip()


## Phase 3 — RECOMMENDED: independent context-sample replication

Buckets 2 and 3 are disjoint from bucket 1. At the primary depth this uses **31 alternative receivers** per fixed update (32 including the original-host QA anchor).


In [ ]:
# ============================================================
# PHASE 3 — BUCKET 2, 32 RECEIVERS / DELTA, BOTH MODELS
# ============================================================
if phase_ok(2.5, "Bucket-2 primary-depth replication"):
    for model_key in ["mistral", "qwen"]:
        spec = MODEL_SPECS[model_key]
        model = tokenizer = None
        try:
            model, tokenizer = load_model(spec)
            run_cross_stage(
                model, tokenizer,
                model_key=model_key,
                bucket=2,
                rel_depth=0.40,
                receivers_per_delta=cfg.receivers_replication,
            )
        finally:
            unload_model(model, tokenizer)
        write_checkpoint_zip()


In [ ]:
# ============================================================
# PHASE 4 — BUCKET 3, 32 RECEIVERS / DELTA, BOTH MODELS
# Automatically skips if the 15h soft wall would be exceeded.
# ============================================================
if phase_ok(2.5, "Bucket-3 primary-depth replication"):
    for model_key in ["mistral", "qwen"]:
        spec = MODEL_SPECS[model_key]
        model = tokenizer = None
        try:
            model, tokenizer = load_model(spec)
            run_cross_stage(
                model, tokenizer,
                model_key=model_key,
                bucket=3,
                rel_depth=0.40,
                receivers_per_delta=cfg.receivers_replication,
            )
        finally:
            unload_model(model, tokenizer)
        write_checkpoint_zip()


In [ ]:
# ============================================================
# CONSOLIDATED SCIENTIFIC SUMMARY
# ============================================================
summaries = []
for p in OUT_ROOT.rglob("stage_summary.json"):
    try:
        d = json.loads(p.read_text())
        d["path"] = str(p.relative_to(OUT_ROOT))
        summaries.append(d)
    except Exception as e:
        print("Could not read", p, e)

summary_df = pd.DataFrame(summaries)
summary_df.to_csv(OUT_ROOT / "ALL_STAGE_SUMMARIES.csv", index=False)

if len(summary_df):
    if "fisher_median_alt_rho" in summary_df.columns:
        cross_df = summary_df[summary_df["fisher_median_alt_rho"].notna()].copy()
        if len(cross_df):
            print("\n=== PHYSICAL CROSSED-RECEIVER RESULTS ===")
            cols = [
                "model_key", "bucket", "layer_idx", "relative_depth",
                "receivers_per_delta", "fisher_pooled_rho",
                "fisher_median_alt_rho", "fisher_positive_delta_fraction",
                "best_simple", "best_simple_median_alt_rho",
                "fisher_minus_best_simple_alt_rho",
            ]
            display(cross_df[[c for c in cols if c in cross_df.columns]].sort_values(["model_key","bucket","layer_idx"]))

    if "fisher_two_way_resid_rho" in summary_df.columns:
        core_df = summary_df[summary_df["fisher_two_way_resid_rho"].notna()].copy()
        if len(core_df):
            print("\n=== FULL-CORE RESULTS ===")
            cols = [
                "model_key", "bucket", "layer_idx", "n_pairs",
                "fisher_raw_rho", "fisher_two_way_resid_rho",
                "best_simple", "best_simple_two_way_resid_rho",
                "fisher_minus_best_simple_resid_rho",
            ]
            display(core_df[[c for c in cols if c in core_df.columns]])

print("\nElapsed wall-clock hours:", round(elapsed_hours(), 2))
disk_status("Before final packaging")
gpu_status("Before final packaging")


In [ ]:
# ============================================================
# FINAL PACKAGE
# ============================================================
clear_model_cache()
clear_dataset_cache()
disk_status("After cache cleanup")

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_ROOT.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(OUT_ROOT))

print("\nFINAL RESULTS:", FINAL_ZIP)
print(f"Size: {FINAL_ZIP.stat().st_size/2**20:.1f} MB")
print("CHECKPOINT:", CHECKPOINT_ZIP)
print("Download BOTH ZIPs before stopping the Kaggle session.")


## What to upload back here

- `/kaggle/working/RECEIVER_CAUSAL_T4X2_V3_RESULTS.zip`
- `/kaggle/working/RECEIVER_CAUSAL_T4X2_V3_CHECKPOINT.zip`

### Stop conditions
- If either T4 already has >0.75 GiB allocated before model loading, restart the session.
- If a loaded model uses >10.5 GiB on one T4, stop that phase and send the device map/memory output.
- Do **not** retry exact JVP on these T4s.
- Do **not** start 14B/32B before the Mistral physical receiver-crossing replication is secured.
